In [1]:
import digitalhub as dh
import pandas as pd
import requests
import os

In [2]:
modes: list[str] = ["CAR"]  ## Choose one or more among: "car_park", "car", "walk", "transit
zoi: str = "restrictedAv"  ## Choose between "allBologna" and "restrictedAv"
method: str = "simplified" ## Choose between "simplified" and "extended"
version: str = "20250926"

In [3]:
if zoi not in ["allBologna", "restrictedAv"]:
    raise ValueError("zoi is not acceptable")
if method not in ["simplified", "extended"]:
    raise ValueError("method is not acceptable")
for mode in modes:
    if mode not in ["CAR_PARK", "TRANSIT", "WALK", "CAR"]:
        raise ValueError("one of modes is not acceptable")

## Prepare OTP
Define function, build, run server

In [4]:
# Project
PROJECT = "test-otp-v2"
project = dh.get_or_create_project(PROJECT)

In [5]:
# OTP server function
func = project.new_function(
    name=f"otp-server-{zoi}",
    kind="container",
    base_image="docker.io/opentripplanner/opentripplanner:latest")

In [6]:
# Go to the Minio console, and create shareable links for each of the artifacts (pbf and gtfs)
GTFS = "http://minio.bolognadigitaltwin.it/api/v1/download-shared-object/aHR0cDovLzEyNy4wLjAuMTo5MDAwL2RhdGFsYWtlL3Rlc3Qtb3RwLXYyL2FydGlmYWN0Lzl1OW41bDYxeHVkZmY1YmRtZm1hMXptMC9ndGZzX2dvbW1hZ3Rmc2JvXzIwMjUwNTEzLnppcD9YLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPVFLMEtPWEEwQTJYRThROVFHMTFIJTJGMjAyNTA5MjYlMkZ1cy1lYXN0LTElMkZzMyUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjUwOTI2VDEzNTUwNVomWC1BbXotRXhwaXJlcz00MzE5OSZYLUFtei1TZWN1cml0eS1Ub2tlbj1leUpoYkdjaU9pSklVelV4TWlJc0luUjVjQ0k2SWtwWFZDSjkuZXlKaFkyTmxjM05MWlhraU9pSlJTekJMVDFoQk1FRXlXRVU0VVRsUlJ6RXhTQ0lzSW1GMFgyaGhjMmdpT2lKVmF6ZGtaVEJMV1ZaTlJEWnRWVkJNVDB0WlRGWkJJaXdpWVhWa0lqb2lXVEk1U1ZkSVFsRldWWGhWWVVSR2JGTjZSbEJPUmxKdlRXdGFSbEl4YUVraUxDSmhkWFJvWDNScGJXVWlPakUzTlRnNE9EWTFOallzSW1GMWRHaHZjbWwwYVdWeklqcGJYU3dpWVhwd0lqb2lXVEk1U1ZkSVFsRldWWGhWWVVSR2JGTjZSbEJPUmxKdlRXdGFSbEl4YUVraUxDSmxiV0ZwYkNJNkltRmlkWEo2WVdOamFHbEFabUpyTG1WMUlpd2laVzFoYVd4ZmRtVnlhV1pwWldRaU9uUnlkV1VzSW1WNGNDSTZNVGMxT0RreU9UYzJOaXdpWm1GdGFXeDVYMjVoYldVaU9pSkNkWEo2WVdOamFHa2lMQ0puYVhabGJsOXVZVzFsSWpvaVFYSnBZVzV1WVNJc0ltbGhkQ0k2TVRjMU9EZzROalUyT0N3aWFXUWlPaUp2Y0dWdWFXUWlMQ0pwYzNNaU9pSm9kSFJ3Y3pvdkwyRmhZeTVpYjJ4dloyNWhaR2xuYVhSaGJIUjNhVzR1YVhRaUxDSnFkR2tpT2lKSldFeEVVbWxWZW5CQlR6ZFFUakpXVEV4TlpuWkVUMmhaZUUwaUxDSnNiMk5oYkdVaU9pSmxiaUlzSW01aVppSTZNVGMxT0RnNE5qVTJPQ3dpY0dsamRIVnlaU0k2SW1oMGRIQnpPaTh2YkdnekxtZHZiMmRzWlhWelpYSmpiMjUwWlc1MExtTnZiUzloTDBGRFp6aHZZMHBNVDFoc2VFRjNkMWRNYzA1UldWSkNVRVZFYjNnd1MzRXRZMUZOWVdScGEwb3pXVloyWVdoRk5VeGhhVlo1VVhoRFBYTTVOaTFqSWl3aWNHOXNhV041SWpvaWNtVmhaSGR5YVhSbFpHbG5hWFJoYkdoMVlpSXNJbkJ5WldabGNuSmxaRjkxYzJWeWJtRnRaU0k2SW1GaWRYSjZZV05qYUdsQVptSnJMbVYxSWl3aWNtVmhiRzBpT2lKa2FXZHBkR0ZzYUhWaUxXSnZiRzluYm1FaUxDSnliMnhsY3lJNlcxMHNJbk53WVdObFVtOXNaWE1pT2x0ZExDSnpkV0lpT2lKMVgySTFZMll6WldabE1EQmhZelEyTkRjNVlURmhaalpqTWpobFkyRXdNV00zSWl3aWVtOXVaV2x1Wm04aU9pSkhUVlFpZlEuNWRpV01NeGZzYmFWVl9yMU1BdlliaE9YT3F0N0xRQkdGVVlzaDAzLTFXaEVsNnN3d3lINVJrNmN6NWNCT1VpdzlBZFk4V29pZjA3b1JIYzczTEhsUGcmWC1BbXotU2lnbmVkSGVhZGVycz1ob3N0JnZlcnNpb25JZD1udWxsJlgtQW16LVNpZ25hdHVyZT1iY2UxMWQ0ZjU4NmU4NTBjMjBmMzQzYjFjYWQxOTc0OTA3ODAyMDExNmRkZTc3ZDg5ODAwNGVhMjhjODQ0OWI0"
if zoi == "allBologna":
    PBF = "http://minio.bolognadigitaltwin.it/api/v1/download-shared-object/aHR0cDovLzEyNy4wLjAuMTo5MDAwL2RhdGFsYWtlL3Rlc3Qtb3RwLXYyL2FydGlmYWN0L3hwd2lkaTVyMXE5ZjAzMG1mbWE0dXJtL2JvbG9nbmEtYXJlYS1maWx0ZXJlZC1wYXJraW5nLXNvcnRlZC5vc20ucGJmP1gtQW16LUFsZ29yaXRobT1BV1M0LUhNQUMtU0hBMjU2JlgtQW16LUNyZWRlbnRpYWw9UUswS09YQTBBMlhFOFE5UUcxMUglMkYyMDI1MDkyNiUyRnVzLWVhc3QtMSUyRnMzJTJGYXdzNF9yZXF1ZXN0JlgtQW16LURhdGU9MjAyNTA5MjZUMTM1NTU0WiZYLUFtei1FeHBpcmVzPTQzMjAwJlgtQW16LVNlY3VyaXR5LVRva2VuPWV5SmhiR2NpT2lKSVV6VXhNaUlzSW5SNWNDSTZJa3BYVkNKOS5leUpoWTJObGMzTkxaWGtpT2lKUlN6QkxUMWhCTUVFeVdFVTRVVGxSUnpFeFNDSXNJbUYwWDJoaGMyZ2lPaUpWYXpka1pUQkxXVlpOUkRadFZWQk1UMHRaVEZaQklpd2lZWFZrSWpvaVdUSTVTVmRJUWxGV1ZYaFZZVVJHYkZONlJsQk9SbEp2VFd0YVJsSXhhRWtpTENKaGRYUm9YM1JwYldVaU9qRTNOVGc0T0RZMU5qWXNJbUYxZEdodmNtbDBhV1Z6SWpwYlhTd2lZWHB3SWpvaVdUSTVTVmRJUWxGV1ZYaFZZVVJHYkZONlJsQk9SbEp2VFd0YVJsSXhhRWtpTENKbGJXRnBiQ0k2SW1GaWRYSjZZV05qYUdsQVptSnJMbVYxSWl3aVpXMWhhV3hmZG1WeWFXWnBaV1FpT25SeWRXVXNJbVY0Y0NJNk1UYzFPRGt5T1RjMk5pd2labUZ0YVd4NVgyNWhiV1VpT2lKQ2RYSjZZV05qYUdraUxDSm5hWFpsYmw5dVlXMWxJam9pUVhKcFlXNXVZU0lzSW1saGRDSTZNVGMxT0RnNE5qVTJPQ3dpYVdRaU9pSnZjR1Z1YVdRaUxDSnBjM01pT2lKb2RIUndjem92TDJGaFl5NWliMnh2WjI1aFpHbG5hWFJoYkhSM2FXNHVhWFFpTENKcWRHa2lPaUpKV0V4RVVtbFZlbkJCVHpkUVRqSldURXhOWm5aRVQyaFplRTBpTENKc2IyTmhiR1VpT2lKbGJpSXNJbTVpWmlJNk1UYzFPRGc0TmpVMk9Dd2ljR2xqZEhWeVpTSTZJbWgwZEhCek9pOHZiR2d6TG1kdmIyZHNaWFZ6WlhKamIyNTBaVzUwTG1OdmJTOWhMMEZEWnpodlkwcE1UMWhzZUVGM2QxZE1jMDVSV1ZKQ1VFVkViM2d3UzNFdFkxRk5ZV1JwYTBveldWWjJZV2hGTlV4aGFWWjVVWGhEUFhNNU5pMWpJaXdpY0c5c2FXTjVJam9pY21WaFpIZHlhWFJsWkdsbmFYUmhiR2gxWWlJc0luQnlaV1psY25KbFpGOTFjMlZ5Ym1GdFpTSTZJbUZpZFhKNllXTmphR2xBWm1KckxtVjFJaXdpY21WaGJHMGlPaUprYVdkcGRHRnNhSFZpTFdKdmJHOW5ibUVpTENKeWIyeGxjeUk2VzEwc0luTndZV05sVW05c1pYTWlPbHRkTENKemRXSWlPaUoxWDJJMVkyWXpaV1psTURCaFl6UTJORGM1WVRGaFpqWmpNamhsWTJFd01XTTNJaXdpZW05dVpXbHVabThpT2lKSFRWUWlmUS41ZGlXTU14ZnNiYVZWX3IxTUF2WWJoT1hPcXQ3TFFCR0ZVWXNoMDMtMVdoRWw2c3d3eUg1Ums2Y3o1Y0JPVWl3OUFkWThXb2lmMDdvUkhjNzNMSGxQZyZYLUFtei1TaWduZWRIZWFkZXJzPWhvc3QmdmVyc2lvbklkPW51bGwmWC1BbXotU2lnbmF0dXJlPWMwODFkOTkzOWFkZTAzMDU3MjQ4OGNhNTJjYzNlNDhlNTA3MzZkOWM2MjVlYmQ3MzBiYmQ2OTA4ZGIyNDU2ZTE"
elif zoi == "restrictedAv":
    PBF1 = "http://minio.bolognadigitaltwin.it/api/v1/download-shared-object/aHR0cDovLzEyNy4wLjAuMTo5MDAwL2RhdGFsYWtlL3Rlc3Qtb3RwLXYyL2FydGlmYWN0L2hzbnljbnR5aW53ZTJlNWJtZzB2eWM0MS9ib2xvZ25hLWFyZWEtZmlsdGVyZWQtcGFya2luZy1pbnNpZGUtQVYtZm9vdHdheS5vc20ucGJmP1gtQW16LUFsZ29yaXRobT1BV1M0LUhNQUMtU0hBMjU2JlgtQW16LUNyZWRlbnRpYWw9UUswS09YQTBBMlhFOFE5UUcxMUglMkYyMDI1MDkyNiUyRnVzLWVhc3QtMSUyRnMzJTJGYXdzNF9yZXF1ZXN0JlgtQW16LURhdGU9MjAyNTA5MjZUMTM1NTM1WiZYLUFtei1FeHBpcmVzPTQzMjAwJlgtQW16LVNlY3VyaXR5LVRva2VuPWV5SmhiR2NpT2lKSVV6VXhNaUlzSW5SNWNDSTZJa3BYVkNKOS5leUpoWTJObGMzTkxaWGtpT2lKUlN6QkxUMWhCTUVFeVdFVTRVVGxSUnpFeFNDSXNJbUYwWDJoaGMyZ2lPaUpWYXpka1pUQkxXVlpOUkRadFZWQk1UMHRaVEZaQklpd2lZWFZrSWpvaVdUSTVTVmRJUWxGV1ZYaFZZVVJHYkZONlJsQk9SbEp2VFd0YVJsSXhhRWtpTENKaGRYUm9YM1JwYldVaU9qRTNOVGc0T0RZMU5qWXNJbUYxZEdodmNtbDBhV1Z6SWpwYlhTd2lZWHB3SWpvaVdUSTVTVmRJUWxGV1ZYaFZZVVJHYkZONlJsQk9SbEp2VFd0YVJsSXhhRWtpTENKbGJXRnBiQ0k2SW1GaWRYSjZZV05qYUdsQVptSnJMbVYxSWl3aVpXMWhhV3hmZG1WeWFXWnBaV1FpT25SeWRXVXNJbVY0Y0NJNk1UYzFPRGt5T1RjMk5pd2labUZ0YVd4NVgyNWhiV1VpT2lKQ2RYSjZZV05qYUdraUxDSm5hWFpsYmw5dVlXMWxJam9pUVhKcFlXNXVZU0lzSW1saGRDSTZNVGMxT0RnNE5qVTJPQ3dpYVdRaU9pSnZjR1Z1YVdRaUxDSnBjM01pT2lKb2RIUndjem92TDJGaFl5NWliMnh2WjI1aFpHbG5hWFJoYkhSM2FXNHVhWFFpTENKcWRHa2lPaUpKV0V4RVVtbFZlbkJCVHpkUVRqSldURXhOWm5aRVQyaFplRTBpTENKc2IyTmhiR1VpT2lKbGJpSXNJbTVpWmlJNk1UYzFPRGc0TmpVMk9Dd2ljR2xqZEhWeVpTSTZJbWgwZEhCek9pOHZiR2d6TG1kdmIyZHNaWFZ6WlhKamIyNTBaVzUwTG1OdmJTOWhMMEZEWnpodlkwcE1UMWhzZUVGM2QxZE1jMDVSV1ZKQ1VFVkViM2d3UzNFdFkxRk5ZV1JwYTBveldWWjJZV2hGTlV4aGFWWjVVWGhEUFhNNU5pMWpJaXdpY0c5c2FXTjVJam9pY21WaFpIZHlhWFJsWkdsbmFYUmhiR2gxWWlJc0luQnlaV1psY25KbFpGOTFjMlZ5Ym1GdFpTSTZJbUZpZFhKNllXTmphR2xBWm1KckxtVjFJaXdpY21WaGJHMGlPaUprYVdkcGRHRnNhSFZpTFdKdmJHOW5ibUVpTENKeWIyeGxjeUk2VzEwc0luTndZV05sVW05c1pYTWlPbHRkTENKemRXSWlPaUoxWDJJMVkyWXpaV1psTURCaFl6UTJORGM1WVRGaFpqWmpNamhsWTJFd01XTTNJaXdpZW05dVpXbHVabThpT2lKSFRWUWlmUS41ZGlXTU14ZnNiYVZWX3IxTUF2WWJoT1hPcXQ3TFFCR0ZVWXNoMDMtMVdoRWw2c3d3eUg1Ums2Y3o1Y0JPVWl3OUFkWThXb2lmMDdvUkhjNzNMSGxQZyZYLUFtei1TaWduZWRIZWFkZXJzPWhvc3QmdmVyc2lvbklkPW51bGwmWC1BbXotU2lnbmF0dXJlPWJjMTdiZDg0ZDY2Njg2ZGNlYjg4NjkwNmE5YzY3ZjA2YzU5YTRmN2M5MWZlZTgzNjg2NmM4ZGU0MjM5M2EwOWI"
    PBF2 = "http://minio.bolognadigitaltwin.it/api/v1/download-shared-object/aHR0cDovLzEyNy4wLjAuMTo5MDAwL2RhdGFsYWtlL3Rlc3Qtb3RwLXYyL2FydGlmYWN0L2Izcm90MXoyNjFsMWM0ZGJtZm5uejBvdS9ib2xvZ25hLWFyZWEtZmlsdGVyZWQtcGFya2luZy1vdXRzaWRlLUFWLm9zbS5wYmY_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1RSzBLT1hBMEEyWEU4UTlRRzExSCUyRjIwMjUwOTI2JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MDkyNlQxMzU1MjBaJlgtQW16LUV4cGlyZXM9NDMyMDAmWC1BbXotU2VjdXJpdHktVG9rZW49ZXlKaGJHY2lPaUpJVXpVeE1pSXNJblI1Y0NJNklrcFhWQ0o5LmV5SmhZMk5sYzNOTFpYa2lPaUpSU3pCTFQxaEJNRUV5V0VVNFVUbFJSekV4U0NJc0ltRjBYMmhoYzJnaU9pSlZhemRrWlRCTFdWWk5SRFp0VlZCTVQwdFpURlpCSWl3aVlYVmtJam9pV1RJNVNWZElRbEZXVlhoVllVUkdiRk42UmxCT1JsSnZUV3RhUmxJeGFFa2lMQ0poZFhSb1gzUnBiV1VpT2pFM05UZzRPRFkxTmpZc0ltRjFkR2h2Y21sMGFXVnpJanBiWFN3aVlYcHdJam9pV1RJNVNWZElRbEZXVlhoVllVUkdiRk42UmxCT1JsSnZUV3RhUmxJeGFFa2lMQ0psYldGcGJDSTZJbUZpZFhKNllXTmphR2xBWm1KckxtVjFJaXdpWlcxaGFXeGZkbVZ5YVdacFpXUWlPblJ5ZFdVc0ltVjRjQ0k2TVRjMU9Ea3lPVGMyTml3aVptRnRhV3g1WDI1aGJXVWlPaUpDZFhKNllXTmphR2tpTENKbmFYWmxibDl1WVcxbElqb2lRWEpwWVc1dVlTSXNJbWxoZENJNk1UYzFPRGc0TmpVMk9Dd2lhV1FpT2lKdmNHVnVhV1FpTENKcGMzTWlPaUpvZEhSd2N6b3ZMMkZoWXk1aWIyeHZaMjVoWkdsbmFYUmhiSFIzYVc0dWFYUWlMQ0pxZEdraU9pSkpXRXhFVW1sVmVuQkJUemRRVGpKV1RFeE5ablpFVDJoWmVFMGlMQ0pzYjJOaGJHVWlPaUpsYmlJc0ltNWlaaUk2TVRjMU9EZzROalUyT0N3aWNHbGpkSFZ5WlNJNkltaDBkSEJ6T2k4dmJHZ3pMbWR2YjJkc1pYVnpaWEpqYjI1MFpXNTBMbU52YlM5aEwwRkRaemh2WTBwTVQxaHNlRUYzZDFkTWMwNVJXVkpDVUVWRWIzZ3dTM0V0WTFGTllXUnBhMG96V1ZaMllXaEZOVXhoYVZaNVVYaERQWE01Tmkxaklpd2ljRzlzYVdONUlqb2ljbVZoWkhkeWFYUmxaR2xuYVhSaGJHaDFZaUlzSW5CeVpXWmxjbkpsWkY5MWMyVnlibUZ0WlNJNkltRmlkWEo2WVdOamFHbEFabUpyTG1WMUlpd2ljbVZoYkcwaU9pSmthV2RwZEdGc2FIVmlMV0p2Ykc5bmJtRWlMQ0p5YjJ4bGN5STZXMTBzSW5Od1lXTmxVbTlzWlhNaU9sdGRMQ0p6ZFdJaU9pSjFYMkkxWTJZelpXWmxNREJoWXpRMk5EYzVZVEZoWmpaak1qaGxZMkV3TVdNM0lpd2llbTl1WldsdVptOGlPaUpIVFZRaWZRLjVkaVdNTXhmc2JhVlZfcjFNQXZZYmhPWE9xdDdMUUJHRlVZc2gwMy0xV2hFbDZzd3d5SDVSazZjejVjQk9VaXc5QWRZOFdvaWYwN29SSGM3M0xIbFBnJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCZ2ZXJzaW9uSWQ9bnVsbCZYLUFtei1TaWduYXR1cmU9NTM4OTQ2MmM4ZDNjZDBmNTc3NzFkMTZkNjljOWU4ZmQ2ZGQ5Y2U4NWY4MjAyN2NkZTRlZmZmYTA2NTVmZTdiZQ"

In [7]:
# Run the OTP server
if zoi == "allBologna": 
    instructions = [
    "apt-get -y update; apt-get -y install curl",
    "mkdir -p /var/opentripplanner",
    f"curl -L {PBF} -o /var/opentripplanner/osm-tn-bologna.osm.pbf",
    f"curl -L {GTFS} -o /var/opentripplanner/gtfs-pts-bus-bologna.zip"
]
elif zoi == "restrictedAv":
    instructions = [
    "apt-get -y update; apt-get -y install curl",
    "mkdir -p /var/opentripplanner",
    f"curl -L {PBF1} -o /var/opentripplanner/osm-modified-tn-bologna-inside-av.osm.pbf",
    f"curl -L {PBF2} -o /var/opentripplanner/osm-tn-bologna-outside-av.osm.pbf",
    f"curl -L {GTFS} -o /var/opentripplanner/gtfs-pts-bus-bologna.zip"
]

build_run = func.run(action="build", instructions=instructions)
build_status = build_run.wait()
# After running, wait before serving!!!

2025-09-26 13:56:23,984 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:56:29,010 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:56:34,035 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:56:39,059 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:56:44,089 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:56:49,126 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:56:54,161 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:56:59,198 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:57:04,233 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:57:09,263 - INFO - Waiting for run d03c68396827416ca4f87b59318d60c4 to finish...
2025-09-26 13:57:14,289 - INFO - Run d03c683968274

In [8]:
if build_status.status.state == "COMPLETED":
    print(build_status.status.state)
    server_run = func.run(action="serve",
                      args=["--build", "--serve"],
                      service_ports=[{"port": 8080, "target_port": 8080}],
                      service_type='ClusterIP',
                      run_as_user=60000, run_as_group=60000
                    )

COMPLETED


In [ ]:
server_status = server_run.wait()
# After running, wait before serving!!!

## Use OTP

In [10]:
PROJECT = "test-otp-v2"
project = dh.get_or_create_project(PROJECT)

In [ ]:
otp_func = project.new_function("otp-processor",
                                    kind="python",
                                    python_version="PYTHON3_10",
                                    code_src="otp_processor.py",
                                    handler="main_platform")

In [ ]:
if method == "simplified":
    od_input_key = "store://test-otp-v2/dataitem/table/od-coords-simplified:s471xcn8igf703bbmfo4nn3l"
if method == "extended":
    od_input_key = "store://test-otp-v2/dataitem/table/od-coords-extended:19tqn6505owcb31amfv1qz0g"

url_port = "s-containerserve-6adb2a970b884c169c0a724fe89f4d1a.digitalhub-bologna:8080" ### Get it from the RUN>Service>url in DHCore

modes_str =  "_".join(modes)
otp_endpoint = f"http://{url_port}/otp/gtfs/v1"
output_name = f"otp_results_{method}_{zoi}_{modes_str}_v{version}"

In [ ]:
otp_run = otp_func.run("job", 
                       inputs={'df': od_input_key},
                       parameters={'main_modes': modes,
                                   'method': method,
                                   'zoi': zoi,
                                   'output_name': output_name,
                                   'otp_endpoint': otp_endpoint})